In [ ]:
import pandas as pd


In [ ]:
df=pd.read_csv("jackpot.csv",sep=';',engine='python', header=None).fillna(0)
print(df.shape) 
zero = df.columns[(df==0).all()] 
df=df.drop(columns=zero, axis=1) #delete empty column

df_win=df.iloc[:,2:] # remove first two columns
df_win=df_win.applymap(lambda x: 1 if x != 0 else 0)
cols = list(range(1,51))
cols.extend(list(range(1,13)))

print(df_win.shape)
df_win.columns = cols
print(df_win)

(816, 65)
(816, 62)
     1   2   3   4   5   6   7   8   9   ...  4   5   6   7   8   9   10  11  12
0     0   0   0   0   0   0   0   0   0  ...   0   0   1   0   0   0   0   0   0
1     0   0   0   0   0   0   0   0   0  ...   0   0   0   0   0   0   0   1   0
2     1   0   0   0   0   0   0   0   0  ...   0   1   0   0   0   1   0   0   0
3     1   0   0   0   0   0   0   0   0  ...   0   0   0   0   1   0   1   0   0
4     0   1   0   0   0   0   0   0   0  ...   0   0   0   1   0   0   0   0   1
..   ..  ..  ..  ..  ..  ..  ..  ..  ..  ...  ..  ..  ..  ..  ..  ..  ..  ..  ..
811   0   0   0   0   0   0   0   0   0  ...   0   0   0   0   1   0   0   0   0
812   0   0   0   0   1   0   0   0   0  ...   0   0   0   0   0   0   0   0   0
813   0   0   0   0   0   0   1   1   0  ...   1   1   0   0   0   0   0   0   0
814   0   0   0   0   1   0   1   0   0  ...   0   1   0   0   0   0   0   0   0
815   0   0   0   0   1   0   0   1   0  ...   0   0   1   0   1   0   0   0   0

[816 ro

In [ ]:
import torch
win_inputs = torch.tensor(df_win.values, dtype=torch.int)
label_col = torch.ones(win_inputs.shape[0],1, dtype=torch.int)
win_tensor = torch.cat((win_inputs,label_col),dim=1)

fake_inputs_shape = (5000000,62)
fake_inputs = torch.randint(0,2,fake_inputs_shape,dtype=torch.int)
fake_label_col = torch.zeros(fake_inputs.shape[0],1, dtype=torch.int)
fake_tensor = torch.cat((fake_inputs,fake_label_col),dim=1)

traning_data = torch.cat((win_tensor,fake_tensor),dim=0)

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1], dtype=torch.int32)


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Define logistic regression model
class LogisticRegression(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# Prepare data
X = traning_data[:, :-1].float()
y = traning_data[:, -1].float().unsqueeze(1)

model = LogisticRegression(X.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/1000], Loss: 0.1666
Epoch [20/1000], Loss: 0.0287
Epoch [30/1000], Loss: 0.0120
Epoch [40/1000], Loss: 0.0079
Epoch [50/1000], Loss: 0.0063
Epoch [60/1000], Loss: 0.0055
Epoch [70/1000], Loss: 0.0050
Epoch [80/1000], Loss: 0.0046
Epoch [90/1000], Loss: 0.0042
Epoch [100/1000], Loss: 0.0040
Epoch [110/1000], Loss: 0.0037
Epoch [120/1000], Loss: 0.0035
Epoch [130/1000], Loss: 0.0033
Epoch [140/1000], Loss: 0.0031
Epoch [150/1000], Loss: 0.0029
Epoch [160/1000], Loss: 0.0027
Epoch [170/1000], Loss: 0.0026
Epoch [180/1000], Loss: 0.0025
Epoch [190/1000], Loss: 0.0024
Epoch [200/1000], Loss: 0.0022
Epoch [210/1000], Loss: 0.0021
Epoch [220/1000], Loss: 0.0021
Epoch [230/1000], Loss: 0.0020
Epoch [240/1000], Loss: 0.0019
Epoch [250/1000], Loss: 0.0018
Epoch [260/1000], Loss: 0.0018
Epoch [270/1000], Loss: 0.0017
Epoch [280/1000], Loss: 0.0016
Epoch [290/1000], Loss: 0.0016
Epoch [300/1000], Loss: 0.0015
Epoch [310/1000], Loss: 0.0015
Epoch [320/1000], Loss: 0.0014
Epoch [330/1000],

In [ ]:
# Generate a random dataset with 10 million records and 62 features
import torch
random_inputs = torch.randint(0, 2, (10000000, 62), dtype=torch.float)

# Predict probabilities using the trained model
with torch.no_grad():
    probabilities = model(random_inputs)

# Find the index of the highest probability
max_prob_idx = torch.argmax(probabilities).item()
max_prob = probabilities[max_prob_idx].item()

# Get the ticket (row) with the highest probability
best_ticket = random_inputs[max_prob_idx]

# Find columns where the value is 1
columns_with_1 = [i+1 for i, val in enumerate(best_ticket) if val == 1]

print(f'Highest probability: {max_prob:.6f}')
print(f'Columns with value 1 in the highest probability ticket: {columns_with_1}')